# Chapter 8 — RAG Generation
## Topic 2: Citation and Source Attribution

**Run cells in order.**

- Module 1: Setup — context sources + simulated structured tool-call answer
- Module 2: `verify_citations()` — valid vs invalid citation detection
- Module 3: Injection-safety check — citation strings are never used as file/DB lookups
- Module 4: Full pipeline — generate (simulated) -> verify -> route (pass/block)

**Install:** none beyond the standard library.


## Module 1: Setup — Context Sources + Simulated Model Output

**Requires:** nothing

In [1]:
"""
MODULE 1: Setup

CONTEXT_SOURCES: the set of source markers actually sent to the model in
this request's context block (Topic 1's output).

Simulates two model responses via the finalize_answer_with_citations tool
schema -- one well-behaved, one with an invalid citation -- WITHOUT a real
API call, so this module is fully offline-runnable. Swap in a real
client.messages.create(tools=ANSWER_TOOLS, ...) call to replace the
simulation with production behavior.
"""

CONTEXT_SOURCES = {"01_FD_FAQ.pdf", "04_FD_Policy_Reference.pdf", "03_FD_SOPs.pdf"}

ANSWER_TOOLS = [
    {
        "name": "finalize_answer_with_citations",
        "description": "Call this when you have a complete answer, grounded in the provided context.",
        "input_schema": {
            "type": "object",
            "properties": {
                "answer": {"type": "string"},
                "sources_used": {"type": "array", "items": {"type": "string"}},
            },
            "required": ["answer", "sources_used"],
        },
    }
]

# Simulated model output 1: well-formed, all cited sources were genuinely in context
SIMULATED_RESPONSE_GOOD = {
    "answer": "Premature withdrawal incurs a 1 percent penalty on the applicable rate, "
              "as confirmed by both the FAQ and the policy reference.",
    "sources_used": ["01_FD_FAQ.pdf", "04_FD_Policy_Reference.pdf"],
}

# Simulated model output 2: cites a source that was NEVER in this request's context
SIMULATED_RESPONSE_BAD = {
    "answer": "Premature withdrawal incurs a 1 percent penalty, per the internal risk memo.",
    "sources_used": ["01_FD_FAQ.pdf", "internal_risk_memo_2024.pdf"],  # never in context!
}

print(f"Context sources available for this request: {CONTEXT_SOURCES}")
print(f"\nSimulated response 1 (good): {SIMULATED_RESPONSE_GOOD['sources_used']}")
print(f"Simulated response 2 (bad):  {SIMULATED_RESPONSE_BAD['sources_used']}")
print("\nModule 1 complete. Run Module 2.")


Context sources available for this request: {'01_FD_FAQ.pdf', '04_FD_Policy_Reference.pdf', '03_FD_SOPs.pdf'}

Simulated response 1 (good): ['01_FD_FAQ.pdf', '04_FD_Policy_Reference.pdf']
Simulated response 2 (bad):  ['01_FD_FAQ.pdf', 'internal_risk_memo_2024.pdf']

Module 1 complete. Run Module 2.


## Module 2: `verify_citations()` — Valid vs Invalid Detection

**Requires:** Module 1

In [2]:
"""
MODULE 2: Citation Verification

Cheap set-membership check: every cited source must actually appear in
CONTEXT_SOURCES for THIS request. This is a SYNTACTIC check, not a semantic
one -- it does not confirm the CLAIM is true, only that the cited source
was genuinely available (Topic 4 covers the deeper semantic check).
"""

def verify_citations(sources_used: list, context_sources: set) -> dict:
    invalid_citations = [s for s in sources_used if s not in context_sources]
    return {
        "valid": len(invalid_citations) == 0,
        "invalid_citations": invalid_citations,
        "valid_citations": [s for s in sources_used if s in context_sources],
    }


print("=" * 65)
print("Verifying Response 1 (good)")
print("=" * 65)
result_good = verify_citations(SIMULATED_RESPONSE_GOOD["sources_used"], CONTEXT_SOURCES)
print(f"Answer: {SIMULATED_RESPONSE_GOOD['answer']}")
print(f"Valid: {result_good['valid']}")
print(f"Valid citations: {result_good['valid_citations']}")
print(f"Invalid citations: {result_good['invalid_citations']}")

print("\n" + "=" * 65)
print("Verifying Response 2 (bad -- cites a source never in context)")
print("=" * 65)
result_bad = verify_citations(SIMULATED_RESPONSE_BAD["sources_used"], CONTEXT_SOURCES)
print(f"Answer: {SIMULATED_RESPONSE_BAD['answer']}")
print(f"Valid: {result_bad['valid']}")
print(f"Valid citations: {result_bad['valid_citations']}")
print(f"Invalid citations: {result_bad['invalid_citations']}")

assert result_good["valid"] == True, "Response 1 should be valid!"
assert result_bad["valid"] == False, "Response 2 should be invalid!"
print("\n[VERIFIED] Detection logic correctly distinguishes valid from invalid citations.")

print("\nModule 2 complete. Run Module 3.")


Verifying Response 1 (good)
Answer: Premature withdrawal incurs a 1 percent penalty on the applicable rate, as confirmed by both the FAQ and the policy reference.
Valid: True
Valid citations: ['01_FD_FAQ.pdf', '04_FD_Policy_Reference.pdf']
Invalid citations: []

Verifying Response 2 (bad -- cites a source never in context)
Answer: Premature withdrawal incurs a 1 percent penalty, per the internal risk memo.
Valid: False
Valid citations: ['01_FD_FAQ.pdf']
Invalid citations: ['internal_risk_memo_2024.pdf']

[VERIFIED] Detection logic correctly distinguishes valid from invalid citations.

Module 2 complete. Run Module 3.


## Module 3: Injection-Safety — Citations Are Labels, Never Lookups

**Requires:** Module 1, Module 2

In [3]:
"""
MODULE 3: Injection-Safety Demonstration

Shows WHY citation strings from model output must be treated purely as
labels for set-membership checking, NEVER as arguments to a file-system
or database operation -- a citation string is model-generated text and
should be treated with the same caution as any other untrusted input.
"""

# An adversarial "citation" a compromised or manipulated model might produce
ADVERSARIAL_RESPONSE = {
    "answer": "Here is the requested information.",
    "sources_used": ["../../../etc/passwd", "01_FD_FAQ.pdf"],  # path traversal attempt
}

print("Adversarial simulated citation:", ADVERSARIAL_RESPONSE["sources_used"])

# UNSAFE pattern (what NOT to do) -- shown as a comment only, never executed:
# for source in ADVERSARIAL_RESPONSE["sources_used"]:
#     with open(source) as f:   # DANGEROUS: treats model output as a real file path
#         content = f.read()

# SAFE pattern: citation strings are ONLY ever compared against a known,
# pre-computed set -- never used to construct a path, query, or command
safe_result = verify_citations(ADVERSARIAL_RESPONSE["sources_used"], CONTEXT_SOURCES)

print(f"\nSafe verification result: {safe_result}")
print(f"\nThe adversarial path string \'../../../etc/passwd\' was NEVER opened,")
print(f"executed, or used in any file/DB operation -- it was only checked against")
print(f"a known set of legitimate source markers via simple string comparison,")
print(f"and correctly flagged as invalid: {'../../../etc/passwd' in safe_result['invalid_citations']}")

print("\nModule 3 complete. Run Module 4.")


Adversarial simulated citation: ['../../../etc/passwd', '01_FD_FAQ.pdf']

Safe verification result: {'valid': False, 'invalid_citations': ['../../../etc/passwd'], 'valid_citations': ['01_FD_FAQ.pdf']}

The adversarial path string '../../../etc/passwd' was NEVER opened,
executed, or used in any file/DB operation -- it was only checked against
a known set of legitimate source markers via simple string comparison,
and correctly flagged as invalid: True

Module 3 complete. Run Module 4.


## Module 4: Full Pipeline — Generate -> Verify -> Route

**Requires:** Module 1, Module 2, Module 3

In [4]:
"""
MODULE 4: Full Pipeline

Ties citation verification into a pass/block routing decision, mirroring
the theory's recommendation for a regulated domain: invalid citations
block customer-facing surfacing and route to human review, rather than
silently passing through.
"""

def process_rag_response(response: dict, context_sources: set) -> dict:
    """Full decision pipeline: verify citations, then route."""
    verification = verify_citations(response["sources_used"], context_sources)

    if verification["valid"]:
        return {
            "action": "SURFACE_TO_CUSTOMER",
            "answer": response["answer"],
            "verification": verification,
        }
    else:
        return {
            "action": "BLOCK_AND_ROUTE_TO_HUMAN_REVIEW",
            "answer": None,  # never surfaced
            "verification": verification,
            "log_entry": {
                "original_answer": response["answer"],
                "invalid_citations": verification["invalid_citations"],
                "context_available": list(context_sources),
            },
        }


print("=" * 65)
print("Processing Response 1 (good)")
print("=" * 65)
outcome_good = process_rag_response(SIMULATED_RESPONSE_GOOD, CONTEXT_SOURCES)
print(f"Action: {outcome_good['action']}")
print(f"Answer shown to customer: {outcome_good['answer']}")

print("\n" + "=" * 65)
print("Processing Response 2 (bad)")
print("=" * 65)
outcome_bad = process_rag_response(SIMULATED_RESPONSE_BAD, CONTEXT_SOURCES)
print(f"Action: {outcome_bad['action']}")
print(f"Answer shown to customer: {outcome_bad['answer']}")
print(f"Logged for review: {outcome_bad['log_entry']}")

print("\n" + "=" * 65)
print("CHAPTER 8 TOPIC 2 SUMMARY")
print("=" * 65)
print("""
Citation verification: cheap set-membership check -- does each cited
  source actually appear in the context sent for THIS request?
Valid citation != true claim -- this only confirms the source EXISTED
  in context, not that the claim matches what it says (Topic 4's job).
Citation strings are model output -- NEVER use them as file paths,
  DB queries, or commands. Treat as labels for comparison only.
For a regulated domain: invalid citations BLOCK customer-facing
  surfacing and route to human review, logged with full context.

Next: Topic 3 -- Faithfulness vs. Relevance vs. Correctness
""")


Processing Response 1 (good)
Action: SURFACE_TO_CUSTOMER
Answer shown to customer: Premature withdrawal incurs a 1 percent penalty on the applicable rate, as confirmed by both the FAQ and the policy reference.

Processing Response 2 (bad)
Action: BLOCK_AND_ROUTE_TO_HUMAN_REVIEW
Answer shown to customer: None
Logged for review: {'original_answer': 'Premature withdrawal incurs a 1 percent penalty, per the internal risk memo.', 'invalid_citations': ['internal_risk_memo_2024.pdf'], 'context_available': ['01_FD_FAQ.pdf', '04_FD_Policy_Reference.pdf', '03_FD_SOPs.pdf']}

CHAPTER 8 TOPIC 2 SUMMARY

Citation verification: cheap set-membership check -- does each cited
  source actually appear in the context sent for THIS request?
Valid citation != true claim -- this only confirms the source EXISTED
  in context, not that the claim matches what it says (Topic 4's job).
Citation strings are model output -- NEVER use them as file paths,
  DB queries, or commands. Treat as labels for comparison onl